# Steel Surface Defect Detector (YOLO26): training notebook (Colab / Kaggle)

Runs the whole pipeline on a free GPU:
**NEU-DET data -> YOLO format -> train (MLflow) -> evaluate (mAP) -> ONNX export -> predictions figure -> drift reference**.
At the end you download the artifacts and commit them to your GitHub repo; CI/CD takes over from there.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Get the code
Clone your GitHub repo (edit the URL), or upload the project as a zip and `%cd` into it.

In [ ]:
REPO_URL = "https://github.com/<your-username>/defect-detection-mlops.git"  # <- edit me

import os
if not os.path.exists("defect-detection-mlops"):
    !git clone {REPO_URL}
%cd defect-detection-mlops

In [ ]:
# Colab already ships with torch. Ultralytics brings YOLO26 and OpenCV.
!pip install -q "ultralytics>=8.4" mlflow onnx onnxruntime scikit-learn scipy pyyaml pandas kaggle

## 2. Download NEU-DET (steel defects with bounding boxes)
You need **NEU-DET**: images plus Pascal VOC `.xml` annotations (1,800 images, 6 classes).
Upload your `kaggle.json` (Kaggle -> Settings -> API -> Create New Token), then download.

> The dataset slug below is my best guess and may need updating. Search "NEU surface defect database" (or "NEU-DET")
> on Kaggle; any mirror that has the `.xml` annotation files works. The next cell checks that annotations exist.
> On **Kaggle Notebooks**: use *Add Data*, then set `data.raw_dir` in `params.yaml` to `/kaggle/input/<dataset>`.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database -p data/raw --unzip

In [ ]:
# Sanity check: we need both images and .xml annotations
!echo "xml files:" && find data/raw -name "*.xml" | wc -l
!echo "image files:" && find data/raw -type f \( -name "*.jpg" -o -name "*.bmp" -o -name "*.png" \) | wc -l
!find data/raw -name "*.xml" | head -3

## 3. Convert VOC -> YOLO and split (de-duplication, stratified train/val/test)

In [ ]:
!python -m src.prepare_data

In [ ]:
import glob, random
import matplotlib.pyplot as plt
from PIL import Image
from src.prepare_data import CLASSES
from src.visualize import read_gt
from serving.inference import draw_detections

files = random.sample(glob.glob("data/processed/images/train/*"), 6)
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, f in zip(axes.ravel(), files):
    img = Image.open(f).convert("RGB")
    gt = read_gt(f.replace("images", "labels").rsplit(".", 1)[0] + ".txt", CLASSES, img.size)
    ax.imshow(draw_detections(img, gt, CLASSES)); ax.axis("off")
plt.suptitle("Training samples with ground-truth boxes"); plt.tight_layout(); plt.show()

## 4. AUTOPILOT (recommended): one cell does everything, no babysitting
Runs: sweep of cheap variants -> **winner chosen on the validation split** -> refit on train+val -> evaluate on the
test split **once** -> ONNX export (+ parity checks) -> figures -> drift reference -> demo examples.
Variants: `n_640` (recipe), `n_640_seed2` (a noise meter: same recipe, another seed), `n_640_scale` (hypothesis:
less zoom augmentation), `n_960` (higher resolution). Expect roughly the cost of 4 to 5 nano trainings in total.
Don't re-run it hoping the test number goes up: that would leak the test set into your choices.
After it finishes, skip to step 9 (download artifacts). Steps 4b to 8 below are the manual route to the same result.

In [ ]:
# `make` stops at the first failing step (separate `!` lines would keep going). Fewer variants: make best SWEEP="n_640 n_960"
!make best

### Manual route (skip if you used the autopilot)

## 4m. Train (single run)
Starts from **COCO-pretrained `yolo26n.pt`** (transfer learning) and fine-tunes on NEU-DET.
Edit `params.yaml` for other sizes (`yolo26s.pt`, `yolo26m.pt`), `imgsz`, `epochs`. Every run is logged to MLflow.

In [ ]:
!python -m src.train

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["neu-det-yolo"])
runs[["tags.mlflow.runName", "params.train.model", "metrics.final/mAP50B", "metrics.final/mAP50-95B"]]

## 4b. Recommended: sweep a few variants and let the *validation* set pick the winner
`params.yaml -> sweep` lists variants; each trains in its own folder and is logged to the same MLflow experiment.
The winner is chosen on the **validation** split, so the test set stays untouched.
Start with the cheap ones (`n_640` recipe, `n_640_seed2` = a noise meter, `n_640_scale`, `n_960`). Add `n_1280`,
`s_960` only if resolution/size seem to help. Cost relative to `n_640`: n_960 ~2.2x, n_1280 ~4x, s_960 ~8x, l_960 ~34x.
A variant only counts as better if it beats the `n_640` vs `n_640_seed2` gap. If you already ran step 4 you can skip this.

In [ ]:
!python -m src.sweep --only n_640 n_640_seed2 n_640_scale n_960 --promote

## 5. Evaluate on the held-out test set (mAP)

In [ ]:
!python -m src.evaluate
from IPython.display import Image as IPImage, display
display(IPImage("reports/confusion_matrix.png"))

## 6. Export to ONNX (NMS-free) with a parity check, then see predictions from the *exported* model

In [ ]:
!python -m src.export_onnx
!python -m src.visualize
display(IPImage("reports/predictions_grid.png"))

## 7. Drift reference and demo examples

In [ ]:
!python -m src.make_reference
!python -m src.make_examples
!ls -lh models reports metrics examples

## 8. Try the drift monitor (optional)

In [ ]:
!python -m src.simulate_traffic --mode normal --n 200 --reset
!python -m src.monitor || true
!python -m src.simulate_traffic --mode drifted --n 200 --reset
!python -m src.monitor || true

## 9. Download artifacts
Copy these into your local repo (same paths), then `git add`, commit and push:
`models/model.onnx`, `models/classes.json`, `metrics/*.json`, `reports/*`, `examples/*`.

In [ ]:
!zip -qr artifacts.zip models/model.onnx models/classes.json metrics reports examples
files.download("artifacts.zip")